In [6]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.decomposition import PCA
import json, pickle

In [4]:
# ── paths ──────────────────────────────────────────────────────────────────────
ROOT = Path('../..').resolve()
ANALYSIS = ROOT / "analysis" / "affective_subspace_coverage"
ACTIVATION = ROOT / "activation" / "emotion_rewrites"
FIGURES = ROOT / "thesis" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

# ── colour / style ─────────────────────────────────────────────────────────────
EMOTION_COLOURS = {
    "joy": "#F4C542",
    "trust": "#5BAD6F",
    "fear": "#7B5EA7",
    "surprise": "#F08030",
    "sadness": "#5B8DB8",
    "disgust": "#8B5E3C",
    "anger": "#D94040",
    "anticipation": "#E07840",
}
CANDIDATE_LAYERS = [10, 13, 16]

plt.rcParams.update(
    {
        "font.family": "serif",
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

In [5]:
def plot_layer_lineplot(save_path: Path) -> None:
    df = pd.read_csv(ANALYSIS / "layer_comparison_headline.csv")
    df = df.sort_values("layer")

    metrics = [
        ("emo_probe_bacc_pc4",      "Emotion-category probe\nbalanced accuracy (4-D subspace)"),
        ("centroid_evr4",           r"Centroid compactness EVR$_4$"),
        ("mean_local_pc1_int_rho",  "Mean local PC1 –\nintensity Spearman ρ"),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(10, 3.2), sharey=False)

    for ax, (col, label) in zip(axes, metrics):
        ax.plot(df["layer"], df[col], marker="o", color="#2C5F8A", linewidth=1.8,
                markersize=6, zorder=3)

        # highlight layer 13
        val13 = float(df.loc[df["layer"] == 13, col].iloc[0])
        ax.axvline(13, color="#D94040", linewidth=0.8, linestyle="--", zorder=2,
                   label="Layer 13")
        ax.scatter([13], [val13], color="#D94040", zorder=4, s=50)

        ax.set_xlabel("Layer")
        ax.set_ylabel(label, labelpad=4)
        ax.set_xticks(df["layer"].tolist())
        ax.tick_params(axis="both", which="major", labelsize=9)

    axes[0].legend(fontsize=8, loc="lower right")
    fig.suptitle(
        "Layer-wise diagnostics for affective residual representations",
        fontsize=11,
        y=1.02,
    )
    fig.tight_layout()
    fig.savefig(save_path, bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"Saved: {save_path}")

In [7]:
def load_activations():
    """Return H[N, E, I, L, D] and metadata."""
    info = json.loads((ACTIVATION / "emotion_intensity_residual_stream_info.json").read_text())
    npy_path = ACTIVATION / "emotion_intensity_residual_stream.npy"
    try:
        H = np.load(npy_path, allow_pickle=True)
    except Exception:
        with open(npy_path, "rb") as f:
            H = pickle.load(f)
    if isinstance(H, np.ndarray) and H.dtype == object:
        obj = H.item()
        if isinstance(obj, dict):
            # find the largest array value
            H = max(obj.values(), key=lambda v: v.size if hasattr(v, "size") else 0)
        elif isinstance(obj, np.ndarray):
            H = obj
    return H, info

In [8]:
def plot_centroid_pca(save_path: Path) -> None:
    H, info = load_activations()
    layer_indices: list[int] = info["layer_indices"]   # e.g. [8,10,13,16,19,22]
    emotions: list[str]      = info["emotion_order"]
    # H shape: [N, E, I, L, D]

    target_layers = [10, 13, 16]
    layer_pos = {l: layer_indices.index(l) for l in target_layers}

    # mean over N (source texts) and I (intensities) → centroid per emotion per layer
    # H: [N, E, I, L, D]
    centroids = {}
    for l, lp in layer_pos.items():
        # mean over axis 0 (N) and axis 2 (I)
        c = H[:, :, :, lp, :].mean(axis=(0, 2))  # [E, D]
        centroids[l] = c

    # fit a shared PCA on all centroids concatenated
    all_c = np.concatenate([centroids[l] for l in target_layers], axis=0)  # [3E, D]
    pca = PCA(n_components=2)
    pca.fit(all_c)

    fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))

    for ax, l in zip(axes, target_layers):
        c_2d = pca.transform(centroids[l])  # [E, 2]
        for i, emo in enumerate(emotions):
            colour = EMOTION_COLOURS.get(emo, "#888888")
            ax.scatter(c_2d[i, 0], c_2d[i, 1], color=colour, s=70, zorder=3)
            ax.annotate(
                emo,
                (c_2d[i, 0], c_2d[i, 1]),
                textcoords="offset points",
                xytext=(5, 3),
                fontsize=7.5,
                color=colour,
            )
        ev1 = pca.explained_variance_ratio_[0] * 100
        ev2 = pca.explained_variance_ratio_[1] * 100
        ax.set_xlabel(f"PC1 ({ev1:.1f}%)", fontsize=9)
        ax.set_ylabel(f"PC2 ({ev2:.1f}%)", fontsize=9)
        ax.set_title(f"Layer {l}", fontsize=10, fontweight="bold")
        ax.axhline(0, color="#cccccc", linewidth=0.6, zorder=1)
        ax.axvline(0, color="#cccccc", linewidth=0.6, zorder=1)
        ax.spines[["top", "right"]].set_visible(False)

    fig.suptitle(
        "PCA of emotion centroids (shared basis) at candidate layers",
        fontsize=11,
        y=1.02,
    )
    fig.tight_layout()
    fig.savefig(save_path, bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"Saved: {save_path}")

In [9]:
def generate_latex_table() -> str:
    df = pd.read_csv(ANALYSIS / "layer_comparison_headline.csv")
    df = df[df["layer"].isin(CANDIDATE_LAYERS)].sort_values("layer")

    rows = []
    best = {
        "emo_probe_bacc_pc4": df["emo_probe_bacc_pc4"].max(),
        "centroid_evr4": df["centroid_evr4"].max(),
        "mean_local_pc1_int_rho": df["mean_local_pc1_int_rho"].max(),
    }

    for _, row in df.iterrows():
        layer = int(row["layer"])
        marker = r" \textbf{*}" if layer == 13 else ""

        def fmt(col, fmt_str):
            v = row[col]
            s = fmt_str.format(v)
            if abs(v - best[col]) < 1e-9:
                s = r"\textbf{" + s + "}"
            return s

        rows.append(
            f"  {layer}{marker} & "
            f"{fmt('emo_probe_bacc_pc4', '{:.4f}')} & "
            f"{fmt('centroid_evr4', '{:.4f}')} & "
            f"{fmt('mean_local_pc1_int_rho', '{:.4f}')} \\\\"
        )

    body = "\n".join(rows)
    table = r"""\begin{table}[H]
\centering
\small
\begin{tabular}{lccc}
\toprule
\textbf{Layer} &
  \textbf{Emotion probe BAcc} &
  \textbf{Centroid EVR\textsubscript{4}} &
  \textbf{Mean local PC1 $\rho$} \\
  & \textit{(4-D subspace)} & & \\
\midrule
""" + body + r"""
\bottomrule
\end{tabular}
\caption{Layer-wise diagnostic summary for candidate layers. Emotion-category probe balanced
         accuracy (4-D subspace), centroid compactness (EVR\textsubscript{4}), and mean local
         PC1 intensity correlation. Bold indicates the best value per column; asterisk marks
         the selected reporting layer.}
\label{tab:layer_diagnostics}
\end{table}"""
    return table

In [ ]:
plot_layer_lineplot(FIGURES / "layer_diagnostics_lineplot.pdf")

try:
    plot_centroid_pca(FIGURES / "layer_centroid_pca.pdf")
except Exception as e:
    print(f"[WARN] PCA scatter skipped: {e}")

tex = generate_latex_table()
print("\n── LaTeX table ────────────────────────────────────────────────")
print(tex)
out_path = FIGURES / "layer_diagnostics_table.tex"
out_path.write_text(tex)
print(f"\nSaved: {out_path}")

Saved: /home/maplesugano/proj/EmotionEngine_v2/thesis/figures/layer_diagnostics_lineplot.pdf
[WARN] PCA scatter skipped: invalid load key, '\x00'.

── LaTeX table ────────────────────────────────────────────────
\begin{table}[H]
\centering
\small
\begin{tabular}{lccc}
\toprule
\textbf{Layer} &
  \textbf{Emotion probe BAcc} &
  \textbf{Centroid EVR\textsubscript{4}} &
  \textbf{Mean local PC1 $\rho$} \\
  & \textit{(4-D subspace)} & & \\
\midrule
  10 & 0.6751 & 0.8876 & 0.0998 \\
  13 \textbf{*} & \textbf{0.7033} & 0.8920 & \textbf{0.1336} \\
  16 & 0.6704 & \textbf{0.8958} & 0.1297 \\
\bottomrule
\end{tabular}
\caption{Layer-wise diagnostic summary for candidate layers. Emotion-category probe balanced
         accuracy (4-D subspace), centroid compactness (EVR\textsubscript{4}), and mean local
         PC1 intensity correlation. Bold indicates the best value per column; asterisk marks
         the selected reporting layer.}
\label{tab:layer_diagnostics}
\end{table}

Saved: /home/maple